### 지하철 데이터 전처리

In [ ]:
import os
import pandas as pd
import numpy as np

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 1. 데이터 로드
input_file = os.path.join(DATA_DIR, '지하철 역세권 지가지수.csv')

# 파일 존재 여부 확인
if not os.path.exists(input_file):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {input_file}\ndata 폴더에 원본 CSV 파일을 넣어주세요!")

subway = pd.read_csv(input_file, encoding='cp949')
subway.columns.values[0] = '시도'
subway.columns.values[1] = '시군구'

# 2. 시도 명칭 표준화
sido_map = {
    '서울': '서울특별시', '부산': '부산광역시', '대구': '대구광역시',
    '인천': '인천광역시', '광주': '광주광역시', '대전': '대전광역시',
    '울산': '울산광역시', '경기': '경기도', '강원': '강원도', 
    '충북': '충청북도', '충남': '충청남도', '전북': '전라북도', 
    '전남': '전라남도', '경북': '경상북도', '경남': '경상남도', 
    '제주': '제주특별자치도', '세종': '세종특별자치시'
}
subway['시도'] = subway['시도'].replace(sido_map)

# 3. 부천시 통합 과정
def fix_bucheon(name):
    if pd.isna(name): return name
    # 명칭에 '부천'이 들어있으면 무조건 '부천시'로 통일 (공백 유무 상관없음)
    if '부천' in name:
        return '부천시'
    return name

subway['시군구_통합'] = subway['시군구'].apply(fix_bucheon)

# 4. 중복 제거 및 평균 통합 
# '부천시'로 이름이 같아진 행들 산술평균으로 합침
# '용인시 수지구', '용인시 기흥구' 등은 이름이 다르므로 합쳐지지 않고 유지
date_columns = subway.columns[2:-1] 
subway_wide = subway.groupby(['시도', '시군구_통합'])[date_columns].mean().reset_index()

# 5. 결과 확인 및 저장
output_file = os.path.join(DATA_DIR, 'subway_index_wide_fixed.csv')
subway_wide.to_csv(output_file, index=False, encoding='utf-8-sig')

# 검증 출력
print(f"작업 완료! 파일이 저장되었습니다: {output_file}")
print("--- 통합 결과 확인 (부천시) ---")
print(subway_wide[subway_wide['시군구_통합'] == '부천시'])

--- 통합 결과 확인 (부천시) ---
     시도 시군구_통합  2014년 11월  2014년 12월  2015년 1월  2015년 2월  2015년 3월  2015년 4월  \
13  경기도    부천시        NaN        NaN       NaN       NaN       NaN       NaN   

    2015년 5월  2015년 6월  ...   2024년 12월  2025년 1월    2025년 2월  2025년 3월  \
13       NaN       NaN  ...  101.949667   102.125  102.284333   102.479   

    2025년 4월  2025년 5월  2025년 6월    2025년 7월    2025년 8월  2025년 9월  
13   102.652    102.87   103.081  103.253333  103.426333   103.674  

[1 rows x 133 columns]

--- 유지 결과 확인 (용인시 예시) ---
     시도 시군구_통합  2014년 11월  2014년 12월  2015년 1월  2015년 2월  2015년 3월  2015년 4월  \
30  경기도  용인기흥구     77.353     77.522    77.481    77.473    77.729    77.648   
31  경기도  용인수지구     75.866     75.866    75.866    75.663    75.581    75.568   
32  경기도  용인처인구     73.191     73.263    73.319    73.311    73.373    73.517   

    2015년 5월  2015년 6월  ...  2024년 12월  2025년 1월  2025년 2월  2025년 3월  \
30    77.879    78.001  ...    101.910   101.956   102.184   102.302   
31    75.45

In [ ]:
import os
import pandas as pd

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 1. 지하철 데이터 로드
input_path = os.path.join(DATA_DIR, 'subway_index_wide_fixed.csv')

if not os.path.exists(input_path):
    raise FileNotFoundError(f"이전 단계 파일이 없습니다: {input_path}\n첫 번째 전처리 코드를 먼저 실행해 주세요.")

subway_wide = pd.read_csv(input_path, encoding='utf-8-sig')

# 2. 수동 교정 사전
# 지하철 데이터의 '시군구_통합' 컬럼 값을 기준으로 매핑
correction_map = {
    '고양덕양구': '고양시 덕양구',
    '고양일산동구': '고양시 일산동구',
    '고양일산서구': '고양시 일산서구',
    '성남분당구': '성남시 분당구',
    '성남수정구': '성남시 수정구',
    '성남중원구': '성남시 중원구',
    '수원권선구': '수원시 권선구',
    '수원영통구': '수원시 영통구',
    '수원장안구': '수원시 장안구',
    '수원팔달구': '수원시 팔달구',
    '안산단원구': '안산시 단원구',
    '안산상록구': '안산시 상록구',
    '안양동안구': '안양시 동안구',
    '안양만안구': '안양시 만안구',
    '용인기흥구': '용인시 기흥구',
    '용인수지구': '용인시 수지구',
    '용인처인구': '용인시 처인구',
    '천안동남구': '천안시 동남구',
    '천안서북구': '천안시 서북구'
}

# 3. 사전 적용
subway_wide['시군구_통합'] = subway_wide['시군구_통합'].replace(correction_map)

# 4. 저장 (DATA_DIR 경로 적용)
output_path = os.path.join(DATA_DIR, 'subway_index_wide_fixed_2.csv')
subway_wide.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"수정 완료! 파일 저장 위치: {output_path}")

수정 완료


### 학교 데이터 전처리

In [ ]:
import os
import pandas as pd
import numpy as np

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 1. 데이터 로드 (DATA_DIR 경로 적용)
input_path = os.path.join(DATA_DIR, '전국초중등학교위치표준데이터.csv')

if not os.path.exists(input_path):
    raise FileNotFoundError(f"원본 학교 데이터가 없습니다: {input_path}")

school = pd.read_csv(input_path, encoding='cp949')

# 2. 주소 분리
def split_school_address(address):
    if pd.isna(address):
        return pd.Series([None, None, None])
    
    parts = address.split()
    if len(parts) < 3:
        return pd.Series([None, None, None])
    
    sido = parts[0]
    sigungu = parts[1]
    dong = parts[2]
    
    # 수원시 장안구처럼 '시'와 '구'가 분리되어 있는 경우 처리
    # 주소의 세 번째 단어가 '구'로 끝나면 시군구를 '시+구'로 합치고, 네 번째 단어를 법정동으로 인식
    if len(parts) >= 4 and parts[2].endswith('구'):
        sigungu = f"{parts[1]} {parts[2]}"
        dong = parts[3]
        
    return pd.Series([sido, sigungu, dong])

# 주소 분리 적용
school[['시도', '시군구', '법정동']] = school['소재지지번주소'].apply(split_school_address)

# 3. 시도 명칭 표준화 (지하철 데이터와 동일하게)
sido_map = {
    '서울': '서울특별시', '부산': '부산광역시', '대구': '대구광역시',
    '인천': '인천광역시', '광주': '광주광역시', '대전': '대전광역시',
    '울산': '울산광역시', '경기': '경기도', '강원': '강원도', 
    '충북': '충청북도', '충남': '충청남도', '전북': '전라북도', 
    '전남': '전라남도', '경북': '경상북도', '경남': '경상남도', 
    '제주': '제주특별자치도', '세종': '세종특별자치시'
}
# 학교 데이터 주소는 보통 '서울특별시'처럼 풀네임으로 되어있지만, 만약의 경우를 대비해 사전 적용 (앞글자만 따서 비교하도록 처리)
school['시도'] = school['시도'].apply(lambda x: next((v for k, v in sido_map.items() if k in str(x)), x))

# 4. 부천시 통합 규칙 적용 (주소에 '부천'이 있으면 시군구를 '부천시'로 통일)
school.loc[school['시군구'].str.contains('부천', na=False), '시군구'] = '부천시'

# 5. 지역별 학교급(초/중/고) 개수 집계
school_counts = school.groupby(['시도', '시군구', '법정동', '학교급구분']).size().unstack(fill_value=0).reset_index()

# 컬럼명 정리
school_counts = school_counts.rename(columns={
    '초등학교': '초등학교수', 
    '중학교': '중학교수', 
    '고등학교': '고등학교수'
})

# 합계 계산
school_counts['총학교수'] = school_counts['초등학교수'] + school_counts['중학교수'] + school_counts['고등학교수']

# 6. 결과 저장 (DATA_DIR 경로 적용)
output_path = os.path.join(DATA_DIR, 'school_counts_processed.csv')
school_counts.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"--- 학교 데이터 전처리 완료! 파일 저장 위치: {output_path} ---")
print(school_counts[school_counts['시군구'] == '부천시'].head())

--- 학교 데이터 전처리 완료 (부천시 법정동별 확인) ---
학교급구분   시도  시군구  법정동  고등학교수  중학교수  초등학교수  총학교수
318    경기도  부천시  계수동      0     0      1     1
319    경기도  부천시  고강동      1     1      3     5
320    경기도  부천시  괴안동      0     2      3     5
321    경기도  부천시   내동      0     1      0     1
322    경기도  부천시  도당동      2     1      2     5


시군구 까지만

In [ ]:
import os
import pandas as pd

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 법정동 무시하고 시군구 단위로 다시 합치기
school_counts_sigungu = school_counts.groupby(['시도', '시군구']).agg({
    '초등학교수': 'sum',
    '중학교수': 'sum',
    '고등학교수': 'sum',
    '총학교수': 'sum'
}).reset_index()

# 저장
output_path = os.path.join(DATA_DIR, 'school_counts_sigungu.csv')
school_counts_sigungu.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"시군구 단위 합산 완료! 파일 저장 위치: {output_path}")

# apart_master_final_2026.csv

In [ ]:
import os
import pandas as pd
import numpy as np

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 1. 데이터 로드
print("1/5: 데이터 로드 중...")
# 원본 및 마스터 데이터
apart = pd.read_csv(os.path.join(DATA_DIR, 'Apart Deal.csv'), encoding='cp949', low_memory=False)
master = pd.read_csv(os.path.join(DATA_DIR, '법정동 기준 시군구 단위.csv'), encoding='cp949')

# 이전 단계에서 생성한 데이터
schools = pd.read_csv(os.path.join(DATA_DIR, 'school_counts_sigungu.csv'), encoding='utf-8-sig')
subway = pd.read_csv(os.path.join(DATA_DIR, 'subway_index_wide_fixed_2.csv'), encoding='utf-8-sig')

# 2. 기초 전처리 (날짜 및 지역코드)
print("2/5: 날짜 및 기초 지역코드 정제 중...")
# 날짜 파싱 (다양한 형식 대응)
def ultra_date_parser(x):
    if pd.isna(x) or str(x).strip() == '': return pd.NaT
    val = str(x).strip()
    try:
        if '/' in val:
            parts = val.split(' ')[0].split('/')
            return pd.Timestamp(year=int(parts[2]), month=int(parts[0]), day=1)
        elif '-' in val:
            parts = val.split(' ')[0].split('-')
            return pd.Timestamp(year=int(parts[0]), month=int(parts[1]), day=1)
        return pd.to_datetime(val).replace(day=1)
    except: return pd.NaT

apart['연월_dt'] = apart['거래일'].apply(ultra_date_parser)

# 지역코드 5자리 통일 및 시도 매칭
apart['지역코드_str'] = apart['지역코드'].astype(str).str.split('.').str[0].str.zfill(5)
master['지역코드_5'] = master['시군구_코드_법정동기준'].astype(str).str.split('.').str[0].str.zfill(5)

sido_map = {'11':'서울특별시','26':'부산광역시','27':'대구광역시','28':'인천광역시','29':'광주광역시','30':'대전광역시','31':'울산광역시','36':'세종특별자치시','41':'경기도','42':'강원도','43':'충청북도','44':'충청남도','45':'전라북도','46':'전라남도','47':'경상북도','48':'경상남도','50':'제주특별자치도'}
apart['시도'] = apart['지역코드_str'].str[:2].map(sido_map)

# 3. 시군구 명칭 통합 (학교/지하철 매칭용 커스텀 규칙)
print("3/5: 시군구 명칭 표준화 중...")
master_clean = master[['지역코드_5', '시군구명']].drop_duplicates('지역코드_5')
apart = apart.merge(master_clean, left_on='지역코드_str', right_on='지역코드_5', how='left')

def final_refine_name(row):
    sido = str(row['시도'])
    sgg_raw = str(row['시군구명']).strip()
    if '세종' in sido: return '세종특별자치시'
    parts = sgg_raw.split()
    if sido.endswith('도'): # 도 지역은 시+구 유지
        if parts[0] in sido: return " ".join(parts[1:])
        return sgg_raw
    if sido.endswith('시') and len(parts) > 1: # 광역시는 구 이름만
        return " ".join(parts[1:])
    return sgg_raw

apart['시군구_통합'] = apart.apply(final_refine_name, axis=1)

# 4. 외부 데이터 병합 (학교, 지하철)
print("4/5: 외부 인프라 데이터 병합 중...")
# 학교 데이터 세종시 합산 및 컬럼명 정리
if '시군구' in schools.columns: schools = schools.rename(columns={'시군구': '시군구_통합'})
sejong_sch_val = schools[schools['시도'].str.contains('세종', na=False)]['총학교수'].sum()
schools_fixed = schools[~schools['시도'].str.contains('세종', na=False)].copy()
sejong_row = pd.DataFrame({'시도': ['세종특별자치시'], '시군구_통합': ['세종특별자치시'], '총학교수': [sejong_sch_val]})
schools_fixed = pd.concat([schools_fixed, sejong_row], ignore_index=True)

# 지하철 데이터 Long-format 변환
subway_long = subway.melt(id_vars=['시도', '시군구_통합'], var_name='연월_str', value_name='지하철지수')
subway_long['연월_dt'] = pd.to_datetime(subway_long['연월_str'], format='%Y년 %m월')

# 최종 병합 (중복 컬럼 방지)
df_final = apart.loc[:, ~apart.columns.duplicated()].copy()
df_final = pd.merge(df_final, schools_fixed[['시도', '시군구_통합', '총학교수']].drop_duplicates(['시도', '시군구_통합']), on=['시도', '시군구_통합'], how='left')
df_final = pd.merge(df_final, subway_long[['시도', '시군구_통합', '연월_dt', '지하철지수']].drop_duplicates(['시도', '시군구_통합', '연월_dt']), on=['시도', '시군구_통합', '연월_dt'], how='left')

# 5. 결측치 최종 수동 보정 (지번 & 건축년도)
print("5/5: 최종 결측치 보정 및 저장 중...")
# 지번 보정
jibun_fixes = [("남양읍", "동광뷰엘", "2235"), ("내곡동", "서초포레스타2단지", "384"), ("자곡동", "래미안포레", "687"), ("신원동", "힐스테이트 서초 젠트리스", "557"), ("나성동", "나릿재마을1단지", "835")]
for d, a, j in jibun_fixes:
    df_final.loc[(df_final['법정동'].str.contains(d, na=False)) & (df_final['아파트'] == a), '지번'] = j

# 건축년도 보정
build_fixes = [("조촌동", "더샵디오션시티", 2021), ("중앙동", "힐스테이트속초센트럴", 2021), ("무실동", "더샵원주센트럴파크1단지", 2021), ("가평읍", "가평코아루", 2021), ("오산동", "동탄역유림노르웨이숲", 2021), ("양촌읍", "My더퍼스트", 2021), ("송도동", "더샵송도프라임뷰25BL", 2021), ("신암동", "SG펠리체", 2021)]
for d, a, y in build_fixes:
    df_final.loc[(df_final['법정동'].str.contains(d, na=False)) & (df_final['아파트'] == a), '건축년도'] = y

# 최종 저장
output_path = os.path.join(DATA_DIR, 'apart_master_final_2026.csv')
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n작업 완료! 파일 저장 위치: {output_path}")
print(f"최종 행 수: {len(df_final):,}")

1/5: 데이터 로드 중...
2/5: 날짜 및 기초 지역코드 정제 중...
3/5: 시군구 명칭 표준화 중...
4/5: 외부 인프라 데이터 병합 중...
5/5: 최종 결측치 보정 및 저장 중...

✅ 작업 완료!
최종 행 수: 5,002,839
지번/건축년도 결측치: 0 / 0


### 변수명 변경 및 연식 생성 / apart_master_final_2026_2.csv

In [ ]:
import os
import pandas as pd

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

input_path = os.path.join(DATA_DIR, 'apart_master_final_2026.csv')

final = pd.read_csv(input_path, encoding='utf-8-sig', low_memory=False)

# 1. 변수명 변경 (원하는 영어 이름으로 우측 값을 수정)
# 왼쪽: 현재 이름 / 오른쪽: 바뀔 영어 이름
rename_dict = {
    '법정동': 'dong',
    '아파트': 'apartment_name',
    '지번': 'jibun',
    '전용면적': 'exclusive_area',
    '층': 'floor',
    '건축년도': 'build_year',  # 연식 계산 후 삭제 예정
    '거래금액': 'price',
    '연월_dt': 'transaction_date',
    '시도': 'sido',
    '시군구_통합': 'sigungu',
    '총학교수': 'school_count',
    '지하철지수': 'subway_index'
}

# 변수명 일괄 변경 및 불필요한 컬럼 삭제
df = final.drop(columns=['지역코드', '거래일', '지역코드_str', '지역코드_5', '시군구명']).rename(columns=rename_dict)

# 2. 거래금액(Target) 숫자 변환
df['price'] = pd.to_numeric(df['price'].astype(str).str.replace(',', ''), errors='coerce')

# 3. 날짜형 변환 (이미 datetime인 경우도 대비)
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# 4. '연식(apartment_age)' 변수 생성
# 거래연도(transaction_date.dt.year) - 건축년도
df['apartment_age'] = df['transaction_date'].dt.year - df['build_year'].astype(int)

# 5. 분석에 불필요해진 'build_year' 삭제 (선택 사항)
# 연식을 만들었으므로 원본 건축년도는 삭제하거나 유지 선택
# df = df.drop(columns=['build_year'])

# 6. 나머지 타입 최적화 (Integer & Category)
df['floor'] = pd.to_numeric(df['floor'], errors='coerce').fillna(0).astype(int)
df['school_count'] = df['school_count'].fillna(0).astype(int)

# 반복되는 문자열은 범주형(category)으로 변환
cat_cols = ['dong', 'sido', 'sigungu', 'apartment_name']
for col in cat_cols:
    df[col] = df[col].astype('category')

# 7. 최종 결과 확인
print(df.info())
print(df[['transaction_date', 'apartment_age']].head())

# 8. 최종 저장
output_path = os.path.join(DATA_DIR, 'apart_master_final_2026_2.csv')
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"최적화 완료! 파일 저장 위치: {output_path}")

C:\Users\wooni2004\AppData\Local\Temp\ipykernel_3476\2747540457.py:3: DtypeWarning: Columns (6,8) have mixed types. Specify dtype option on import or set low_memory=False.
  final = pd.read_csv('apart_master_final_2026.csv', encoding='utf-8-sig')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5002839 entries, 0 to 5002838
Data columns (total 13 columns):
 #   Column            Dtype         
---  ------            -----         
 0   dong              category      
 1   apartment_name    category      
 2   jibun             object        
 3   exclusive_area    float64       
 4   floor             int32         
 5   build_year        float64       
 6   price             int64         
 7   transaction_date  datetime64[ns]
 8   sido              category      
 9   sigungu           category      
 10  school_count      int32         
 11  subway_index      float64       
 12  apartment_age     int32         
dtypes: category(4), datetime64[ns](1), float64(3), int32(3), int64(1), object(1)
memory usage: 330.7+ MB
None
  transaction_date  apartment_age
0       2020-05-01             29
1       2020-01-01             14
2       2020-01-01             13
3       2020-01-01             14
4       2020-01-01             13


### 결측치 채우기 + 변수 순서 정렬 / apart_master_final_2026_3.csv

In [ ]:
import os
import pandas as pd

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 이전 단계 데이터 로드 (만약 별도로 실행할 경우를 대비)
# df = pd.read_csv(os.path.join(DATA_DIR, 'apart_master_final_2026_2.csv'), encoding='utf-8-sig')

# 1. build_year 삭제 (이미 연식으로 변환했으므로 제거)
if 'build_year' in df.columns:
    df = df.drop(columns=['build_year'])

# 2. subway_index 결측치 0으로 채우기
# 지하철 지수가 없는 지역은 0으로 간주
df['subway_index'] = df['subway_index'].fillna(0)

# 3. 변수 순서 재배치
new_order = [
    'sido', 'sigungu', 'dong', 'jibun', 'apartment_name', 
    'exclusive_area', 'transaction_date', 'floor', 
    'school_count', 'subway_index', 'apartment_age', 'price'
]

# 지정된 순서로 컬럼 재배치 (혹시 리스트에 없는 컬럼이 있을 경우를 대비해 안전하게 처리)
df = df[new_order]

# '시도', '시군구', '법정동', '지번' 순으로 지역을 묶고, 그 안에서 '거래일'이 오래된 것부터 최신순으로 오름차순 정렬

df = df.sort_values(by=['sido', 'sigungu', 'dong', 'jibun', 'exclusive_area', 'transaction_date'], ascending=True)

# 정렬 후에는 인덱스(행 번호)가 뒤섞이므로 다시 0부터 순서대로 매김
df = df.reset_index(drop=True)

# 4. 최종 확인
print("--- [최종 데이터 구조] ---")
print(df.info())
print("\n--- [데이터 샘플 확인] ---")
display(df.head())

# 5. 파일 저장
output_path = os.path.join(DATA_DIR, 'apart_master_final_2026_3.csv')
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"정렬 및 보정 완료! 파일 저장 위치: {output_path}")

--- [최종 데이터 구조] ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5002839 entries, 0 to 5002838
Data columns (total 12 columns):
 #   Column            Dtype         
---  ------            -----         
 0   sido              category      
 1   sigungu           category      
 2   dong              category      
 3   jibun             object        
 4   apartment_name    category      
 5   exclusive_area    float64       
 6   transaction_date  datetime64[ns]
 7   floor             int32         
 8   school_count      int32         
 9   subway_index      float64       
 10  apartment_age     int32         
 11  price             int64         
dtypes: category(4), datetime64[ns](1), float64(2), int32(3), int64(1), object(1)
memory usage: 292.5+ MB
None

--- [데이터 샘플 확인] ---


,sido,sigungu,dong,jibun,apartment_name,exclusive_area,transaction_date,floor,school_count,subway_index,apartment_age,price
0,강원도,강릉시,강동면 안인리,138-1,대동((구)한주사원),59.96,2015-02-01,1,60,0.0,21,2820
1,강원도,강릉시,강동면 안인리,138-1,대동((구)한주사원),59.96,2015-02-01,2,60,0.0,21,2820
2,강원도,강릉시,강동면 안인리,138-1,대동((구)한주사원),59.96,2015-02-01,3,60,0.0,21,2990
3,강원도,강릉시,강동면 안인리,138-1,대동((구)한주사원),59.96,2015-02-01,4,60,0.0,21,3520
4,강원도,강릉시,강동면 안인리,138-1,대동((구)한주사원),59.96,2015-02-01,4,60,0.0,21,2820


### 달마다 한 개로 샘플링 / apart_master_final_2026_4.csv

In [ ]:
import os
import pandas as pd

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 1. 그룹화 기준 설정
group_cols = [
    'sido', 'sigungu', 'dong', 'jibun', 
    'apartment_name', 'exclusive_area', 'transaction_date'
]

# 2. 평균을 낼 수치형 변수 설정
target_cols = [
    'floor', 'school_count', 'subway_index', 'apartment_age', 'price'
]

print("그룹화 작업 시작 (메모리 최적화 모드)...")

# 3. 메모리 에러 방지용 groupby
# observed=True: 카테고리형 변수에서 데이터에 실제로 존재하는 조합만 계산하여 메모리를 아낌
# as_index=True로 먼저 계산 후 reset_index()를 진행
df_monthly = (
    df.groupby(group_cols, observed=True)[target_cols]
    .mean()
    .reset_index()
)

# 4. CSV 저장 (정렬 후)
print("정렬 및 저장 중...")
df_monthly = df_monthly.sort_values(by=group_cols).reset_index(drop=True)

output_path = os.path.join(DATA_DIR, 'apart_master_final_2026_4.csv')
df_monthly.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"완료! 파일 저장 위치: {output_path}")
print(f"행 수 변화: {len(df):,} -> {len(df_monthly):,}")

그룹화 작업 시작 (메모리 최적화 모드)...
정렬 및 저장 중...
✅ 완료! 행 수 변화: 5,002,839 -> 2,125,400


### 100행으로 변경 및 샘플id 추가 / apart_master_final_2026_5.csv

In [ ]:
import os
import pandas as pd

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 0. 작업 전 원래 타입 저장
original_dtypes = df_monthly.dtypes.to_dict()

# 1. 전체 기간 설정 (2015-01 ~ 2023-04, 총 100개월)
all_months = pd.date_range(start='2015-01-01', end='2023-04-01', freq='MS')
group_keys = ['sido', 'sigungu', 'dong', 'jibun', 'apartment_name', 'exclusive_area']

def reindex_by_month(group):
    # transaction_date를 인덱스로 잡고 100개월로 재지정
    return group.set_index('transaction_date').reindex(all_months)

print("시계열 확장 및 타입 복원 작업 시작...")

# 2. 확장 작업 (Groupby + Reindex)
df_expanded = (
    df_monthly.groupby(group_keys, observed=True)
    .apply(reindex_by_month)
    .drop(columns=group_keys)
    .reset_index()
)

# 3. 컬럼명 정리
df_expanded.rename(columns={'level_6': 'transaction_date'}, inplace=True)

# 4. 그룹 정보(NaN) 채우기
df_expanded[group_keys] = df_expanded[group_keys].ffill().bfill()

# 5. 데이터 타입 강제 복원
for col, dtype in original_dtypes.items():
    if col in df_expanded.columns:
        if pd.api.types.is_integer_dtype(dtype):
            df_expanded[col] = df_expanded[col].astype('float64') 
        else:
            df_expanded[col] = df_expanded[col].astype(dtype)

# sample_id 생성 (그룹별로 고유 번호 부여)
# group_keys가 같은 행들은 동일한 sample_id를 갖게 됨
df_expanded['sample_id'] = df_expanded.groupby(group_keys, observed=True).ngroup()

# 6. 정렬 및 변수 순서 재배치 (sample_id를 맨 앞으로)
cols = ['sample_id'] + [c for c in df_expanded.columns if c != 'sample_id']
df_expanded = df_expanded[cols].sort_values(by=['sample_id', 'transaction_date']).reset_index(drop=True)

# 저장
output_path = os.path.join(DATA_DIR, 'apart_master_final_2026_5.csv')
df_expanded.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"시계열 확장 완료! 파일 저장 위치: {output_path}")
print(f"최종 sample_id 개수: {df_expanded['sample_id'].nunique():,}개")

시계열 확장 및 타입 복원 작업 시작...


C:\Users\wooni2004\AppData\Local\Temp\ipykernel_3476\2290162304.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(reindex_by_month)


50회 이상

In [ ]:
import os
import pandas as pd
import numpy as np

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 1. 데이터 로드
input_path = os.path.join(DATA_DIR, 'apart_master_final_2026_5.csv')
df = pd.read_csv(input_path)

# 2. 시도별로 거래량(sample_id 개수) 상위 10개 시군구 추출
sigungu_counts = df.groupby(['sido', 'sigungu'])['sample_id'].nunique().reset_index()
sigungu_counts.columns = ['sido', 'sigungu', 'sample_count']

# 시도별 top 10 추출 (지방 5대 광역시 및 도 단위 모두 포함)
top_regions = sigungu_counts.groupby('sido').apply(
    lambda x: x.nlargest(10, 'sample_count')
).reset_index(drop=True)

# 3. 지역 식별자(Region Full Name) 생성 및 ID 부여 (Embedding용)
top_regions['region_name'] = top_regions['sido'] + " " + top_regions['sigungu']
region_to_id = {name: i for i, name in enumerate(top_regions['region_name'].unique())}

# 4. 고정 데이터셋 필터링
# 조건 A: 선정된 상위 시군구에 속함
# 조건 B: 해당 아파트 단지(sample_id)의 실제 거래 횟수가 50회 이상
df['region_name'] = df['sido'] + " " + df['sigungu']
sample_transaction_counts = df.groupby('sample_id')['price'].count()
valid_sample_ids = sample_transaction_counts[sample_transaction_counts >= 50].index

df_fixed = df[
    (df['region_name'].isin(top_regions['region_name'])) & 
    (df['sample_id'].isin(valid_sample_ids))
].copy()

# 5. 모델이 읽을 수 있게 지역 ID 컬럼 추가
df_fixed['region_id'] = df_fixed['region_name'].map(region_to_id)

# 6. 최종 파일 저장
output_master = os.path.join(DATA_DIR, 'apart_fixed_master.csv')
output_mapping = os.path.join(DATA_DIR, 'region_mapping.csv')

df_fixed.to_csv(output_master, index=False, encoding='utf-8-sig')

# 매핑 테이블도 저장 (나중에 어떤 ID가 어떤 동네인지 확인용)
pd.DataFrame(list(region_to_id.items()), columns=['region_name', 'region_id']).to_csv(output_mapping, index=False, encoding='utf-8-sig')

print("--- 데이터셋 고정 완료 ---")
print(f"선정된 총 시군구 수: {df_fixed['region_id'].nunique()}개")
print(f"학습에 사용될 총 단지(sample_id) 수: {df_fixed['sample_id'].nunique()}개")
print(f"최종 마스터 저장 위치: {output_master}")
print(f"지역 매핑 정보 저장 위치: {output_mapping}")

C:\Users\wooni2004\AppData\Local\Temp\ipykernel_13684\3920789855.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_regions = sigungu_counts.groupby('sido').apply(


--- 데이터셋 고정 완료 ---
선정된 총 시군구 수: 134개
학습에 사용될 총 단지(sample_id) 수: 7765개
전체 행(Row) 수: 776,500개
파일 저장 완료: apart_fixed_master.csv, region_mapping.csv


In [ ]:
import pandas as pd
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

# 1. 데이터 로드
input_path = os.path.join(DATA_DIR, 'apart_fixed_master.csv')
if not os.path.exists(input_path):
    raise FileNotFoundError(f"마스터 데이터셋이 없습니다: {input_path}")

df = pd.read_csv(input_path, encoding='utf-8-sig')

# 2. 결과 저장용 폴더 생성 (data 폴더 내부에 생성)
output_dir = os.path.join(DATA_DIR, 'sido_split_data')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"새 폴더 생성 완료: {output_dir}")

# 3. sido별로 그룹화하여 파일 저장
sido_list = df['sido'].unique()

print(f"총 {len(sido_list)}개의 시도 데이터를 분리합니다...")

for sido in sido_list:
    # 해당 시도 데이터만 추출
    sido_df = df[df['sido'] == sido]
    
    # 파일명 생성 (예: sido_경기도.csv)
    file_name = f"sido_{sido}.csv"
    file_path = os.path.join(output_dir, file_name)
    
    # 저장 (index=False 권장)
    sido_df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"저장 완료: {file_name} (행 수: {len(sido_df):,})")

print("\n모든 파일 분리가 완료되었습니다!")

총 17개의 시도 데이터를 분리합니다...
저장 완료: sido_강원도.csv (샘플 수: 33000)
저장 완료: sido_경기도.csv (샘플 수: 145700)
저장 완료: sido_경상남도.csv (샘플 수: 57000)
저장 완료: sido_경상북도.csv (샘플 수: 40600)
저장 완료: sido_광주광역시.csv (샘플 수: 48100)
저장 완료: sido_대구광역시.csv (샘플 수: 49900)
저장 완료: sido_대전광역시.csv (샘플 수: 44400)
저장 완료: sido_부산광역시.csv (샘플 수: 44100)
저장 완료: sido_서울특별시.csv (샘플 수: 54600)
저장 완료: sido_세종특별자치시.csv (샘플 수: 4800)
저장 완료: sido_울산광역시.csv (샘플 수: 20300)
저장 완료: sido_인천광역시.csv (샘플 수: 72600)
저장 완료: sido_전라남도.csv (샘플 수: 37100)
저장 완료: sido_전라북도.csv (샘플 수: 45000)
저장 완료: sido_제주특별자치도.csv (샘플 수: 2600)
저장 완료: sido_충청남도.csv (샘플 수: 41000)
저장 완료: sido_충청북도.csv (샘플 수: 35700)

모든 파일 분리가 완료되었습니다!
